In [1]:
import os
import argparse
import random
import logging
import torch

from tqdm import tqdm

import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from torchvision import transforms
from torch.utils.data import DataLoader
from pathlib import Path

from utils import __balance_val_split, __split_of_train_sequence, __log_class_statistics, logger
from datasets.czech_slr_dataset import CzechSLRDataset
from siformer.model import SiFormer, SpoTer
from siformer.utils import train_epoch, evaluate, evaluate_top_k, compute_early_exit_stats, calc_total_params, ConfusionMatrix
from siformer.gaussian_noise import GaussianNoise

from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

import time
import datetime
from statistics import mean

In [2]:

def get_default_args():
    parser = argparse.ArgumentParser(add_help=False)

    parser.add_argument("--experiment_name", type=str, default="WLASL_spoter",
                        help="Name of the experiment after which the logs and plots will be named")
    parser.add_argument("--num_classes", type=int, default=100, help="Number of classes to be recognized by the model")
    parser.add_argument("--batch_size", type=int, default=24, help="Number of batch size")
    parser.add_argument("--num_worker", type=int, default=0, help="Number of workers")
    parser.add_argument("--num_seq_elements", type=int, default=108, # [21(hand)*2 +12(body) ]*2
                        help="Hidden dimension of the underlying Transformer model")
    parser.add_argument("--seed", type=int, default=379,
                        help="Seed with which to initialize all the random components of the training")

    # Data
    parser.add_argument("--training_set_path", type=str, default="", help="Path to the training dataset CSV file")
    parser.add_argument("--testing_set_path", type=str, default="", help="Path to the testing dataset CSV file")
    parser.add_argument("--experimental_train_split", type=float, default=None,
                        help="Determines how big a portion of the training set should be employed (intended for the "
                             "gradually enlarging training set experiment from the paper)")

    parser.add_argument("--validation_set", type=str, choices=["from-file", "split-from-train", "none"],
                        default="none",
                        help="Type of validation set construction. See README for further rederence")
    parser.add_argument("--validation_set_size", type=float,
                        help="Proportion of the training set to be split as validation set, if 'validation_size' is set"
                             " to 'split-from-train'")
    parser.add_argument("--validation_set_path", type=str, default="", help="Path to the validation dataset CSV file")

    # Training hyperparameters
    parser.add_argument("--epochs", type=int, default=100, help="Number of epochs to train the model for")
    parser.add_argument("--lr", type=float, default=0.0001, help="Learning rate for the model training")
    parser.add_argument("--log_freq", type=int, default=1,
                        help="Log frequency (frequency of printing all the training info)")

    # Checkpointing
    parser.add_argument("--save_checkpoints", type=bool, default=True,
                        help="Determines whether to save weights checkpoints")

    # Scheduler
    parser.add_argument("--scheduler_factor", type=int, default=0.1, help="Factor for the ReduceLROnPlateau scheduler")
    parser.add_argument("--scheduler_patience", type=int, default=5,
                        help="Patience for the ReduceLROnPlateau scheduler")

    # Gaussian noise normalization
    parser.add_argument("--gaussian_mean", type=int, default=0, help="Mean parameter for Gaussian noise layer")
    parser.add_argument("--gaussian_std", type=int, default=0.001,
                        help="Standard deviation parameter for Gaussian noise layer")

    # Visualization
    parser.add_argument("--plot_stats", type=bool, default=True,
                        help="Determines whether continuous statistics should be plotted at the end")
    parser.add_argument("--plot_lr", type=bool, default=True,
                        help="Determines whether the LR should be plotted at the end")

    # Training time
    parser.add_argument("--record_training_time", type=bool, default=False,
                        help="Determines whether continuous statistics of training time should be record")

    # Model settings
    parser.add_argument("--attn_type", type=str, default='prob', help="The attention mechanism used by the model")
    parser.add_argument("--num_enc_layers", type=int, default=3, help="Determines the number of encoder layers")
    parser.add_argument("--num_com_layers", type=int, default=1, help="Determines the number of communicating layers")
    parser.add_argument("--num_dec_layers", type=int, default=2, help="Determines the number of decoder layers")
    parser.add_argument("--FIM", type=bool, default=True, help=" ")
    parser.add_argument("--IA_encoder", type=bool, default=True, help="Determines whether input adaptive encoder will be used")
    parser.add_argument("--IA_decoder", type=bool, default=False, help="Determines whether input adaptive decoder will be used")
    parser.add_argument("--pat_enc", type=int, default=1, help="Determines the patience of encoder for earlier exist")
    parser.add_argument("--pat_dec", type=int, default=1, help="Determines the patience of decoder for earlier exist")

    return parser


In [3]:
parser = argparse.ArgumentParser("", parents=[get_default_args()], add_help=False)

parser.set_defaults(
        experiment_name="LSA64",
        training_set_path="datasets/LSA64_60fps.csv",
        experimental_train_split = 0.8,
        validation_set="split-from-train",
        validation_set_size=0.2,
        num_classes=64,
        IA_decoder=True,
        num_worker=2,
        num_com_layers=1,
        num_enc_layers =3,
        num_dec_layers=4,
        pat_enc=1,
        pat_dec=2
    )

args = parser.parse_args(args=[])

In [4]:
random.seed(args.seed)
np.random.seed(args.seed)
os.environ["PYTHONHASHSEED"] = str(args.seed)
torch.manual_seed(args.seed)
torch.cuda.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
torch.backends.cudnn.deterministic = True
g = torch.Generator()
g.manual_seed(args.seed)

In [5]:
logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(args.experiment_name + "_" + str(args.experimental_train_split).replace(".", "") + ".log")
        ]
    )

In [6]:
logger("Experiment parameters:")
logger(f"\t\t\tnum_com_layers={args.num_com_layers}")
logger(f"\t\t\tnum_enc_layers={args.num_enc_layers} | pat_enc={args.pat_enc}")
logger(f"\t\t\tnum_dec_layers={args.num_dec_layers} | pat_enc={args.pat_dec}")

Experiment parameters:
			num_com_layers=1
			num_enc_layers=3 | pat_enc=1
			num_dec_layers=4 | pat_enc=2


In [7]:
device = torch.device("cpu")
if torch.cuda.is_available():
    print("Cuda is available: True")
    device = torch.device("cuda")

Cuda is available: True


In [ ]:
slr_model = SiFormer(num_classes=args.num_classes, num_hid=args.num_seq_elements, attn_type=args.attn_type,
                             num_comm_layers=args.num_com_layers,
                              num_enc_layers=args.num_enc_layers, num_dec_layers=args.num_dec_layers, device=device,
                              IA_encoder=args.IA_encoder, IA_decoder=args.IA_decoder,
                              pat_enc=args.pat_enc, pat_dec=args.pat_dec)

slr_model.train(True)
slr_model.to(device) 

In [9]:
cel_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(slr_model.parameters(), lr=args.lr, betas=(0.9, 0.999), weight_decay=1e-8) # Đây là bộ tối ưu hóa (optimizer). Nó chịu trách nhiệm cập nhật trọng số của mô hình dựa trên giá trị mất mát để cải thiện hiệu suất.
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=0)

In [10]:
Path("out-checkpoints/" + args.experiment_name + "/").mkdir(parents=True, exist_ok=True)
Path("out-img/"+ args.experiment_name + "/").mkdir(parents=True, exist_ok=True)

In [11]:
transform = transforms.Compose([GaussianNoise(args.gaussian_mean, args.gaussian_std)]) #Áp dụng một phép biến đổi lên dữ liệu huấn luyện, cụ thể là thêm nhiễu Gaussian. Đây là một kỹ thuật tăng cường dữ liệu (data augmentation) để giúp mô hình tổng quát hóa tốt hơn.
train_set = CzechSLRDataset(args.training_set_path, transform=transform, augmentations=True) #Đây là một lớp tùy chỉnh để đọc dữ liệu từ các tệp được chỉ định trong args.training_set_path, args.validation_set_path, v.v.

In [ ]:
logger(f'Full dataset size: {len(train_set)} samples')

In [ ]:
train_set, eval_set = __balance_val_split(train_set, 0.2)

logger(f'Training set size: {len(train_set)} samples')
logger(f'Evaluation set size: {len(eval_set)} samples')

train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True, generator=g,
                              num_workers=args.num_worker)

val_loader = None
    
eval_loader = DataLoader(eval_set, batch_size=args.batch_size, shuffle=False, generator=g,
                                 num_workers=args.num_worker)

Training set size: 2541 samples
Evaluation set size: 636 samples


In [14]:
train_acc, val_acc = 0, 0
losses, train_accs, val_accs = [], [], []
lr_progress = []
top_train_acc, top_val_acc = 0, 0
checkpoint_index = 0

In [15]:
if args.experimental_train_split:
    logger(
            "Starting " + args.experiment_name + "_" + str(args.experimental_train_split).replace(".", "") + "...")
else:
    logger("Starting " + args.experiment_name + "...")

logger("Training using " + args.training_set_path + "...")

if args.validation_set == "from-file":
    logger("Validation using " + args.validation_set_path + "...\n\n")

Starting LSA64_08...
Training using datasets/LSA64_60fps.csv...


In [ ]:
total_train_time = 0
avg_train_time_sec_list = []
for epoch in range(args.epochs):
    start_time = time.time()
    train_loss, _, _, train_acc, avg_train_time = train_epoch(slr_model, train_loader, cel_criterion, optimizer,
                                                                  device, scheduler=scheduler)
    end_time = time.time()
    train_time = end_time - start_time

    losses.append(train_loss.item() / len(train_loader))
    train_accs.append(train_acc)

    if args.record_training_time:
        avg_train_time_sec_list.append(avg_train_time)
        total_train_time += train_time

    if val_loader:
        slr_model.train(False)
        _, _, val_acc = evaluate(slr_model, val_loader, device)
        slr_model.train(True)
        val_accs.append(val_acc)

        # Save checkpoints if they are best in the current subset
    if args.save_checkpoints:
        if train_acc > top_train_acc:
            top_train_acc = train_acc
            torch.save(slr_model, "out-checkpoints/" + args.experiment_name + "/checkpoint_t_" + str(
                    checkpoint_index) + ".pth")

        if val_acc > top_val_acc:
            top_val_acc = val_acc
            torch.save(slr_model, "out-checkpoints/" + args.experiment_name + "/checkpoint_v_" + str(
                    checkpoint_index) + ".pth")

            logger(f'Save checkpoint for [{str(epoch + 1)}] as ' + "out-checkpoints/" + args.experiment_name
                      + "/checkpoint_v_" + str(checkpoint_index) + ".pth")

    if epoch % args.log_freq == 0:
        logger(
                "[" + str(epoch + 1) + "] TRAIN  loss: " + str(train_loss.item() / len(train_loader)) + " acc: " + str(
                    train_acc))
        logger(
                f"[{str(epoch + 1)}] AVG TRAIN time per sample (sec): {str(avg_train_time)} "
            )

        if val_loader:
            logger("[" + str(epoch + 1) + "] VALIDATION  acc: " + str(val_acc))

            logger("[" + str(epoch + 1) + "] VALIDATION  Top 5 acc: " + str(top_val_acc))

        logger("")

        # Reset the top accuracies on static subsets
    if epoch % 10 == 0:
        top_train_acc, top_val_acc = 0, 0
        checkpoint_index += 1

    lr_progress.append(optimizer.param_groups[0]["lr"])

[1] TRAIN  loss: 3.821508731482164 acc: 0.1046831955922865
[1] AVG TRAIN time per sample (sec): 0.09751665592193604 

[2] TRAIN  loss: 2.5847320556640625 acc: 0.5757575757575758
[2] AVG TRAIN time per sample (sec): 0.08827131424310072 

[3] TRAIN  loss: 1.8409636875368514 acc: 0.8366784730421094
[3] AVG TRAIN time per sample (sec): 0.08919580927434957 

[4] TRAIN  loss: 1.408587761645047 acc: 0.9126328217237308
[4] AVG TRAIN time per sample (sec): 0.09644355638971869 



In [26]:
if args.record_training_time:
    logger(f"Total training time taken over {args.epochs} epochs: {str(datetime.timedelta(seconds=total_train_time))}")
    logger(f"Average training time per sample: {str(mean(avg_train_time_sec_list[1:]))}")

In [27]:
top_result_top1, top_result_name_top1 = 0, ""
top_result_topk, top_result_name_topk = 0, ""
test_accs_t=[]
test_accs_v=[]
if eval_loader:
        # MARK: TESTING
    logger("\nTesting checkpointed models starting...\n")
    for i in range(11):            
        for checkpoint_id in ["t", "v"]:
            path_to_load = "out-checkpoints/" + args.experiment_name + "/checkpoint_" + checkpoint_id + "_" + str(i) + ".pth"

            if not os.path.exists(path_to_load):                    
                continue               

            tested_model = torch.load(path_to_load, weights_only=False)
            tested_model.eval()

                # === Top 1 ===
            _, _, eval_acc_top1 = evaluate(tested_model, eval_loader, device)

            if checkpoint_id == "v":
                test_accs_v.append(eval_acc_top1)
            else:
                test_accs_t.append(eval_acc_top1)

            if eval_acc_top1 > top_result_top1:
                top_result_top1 = eval_acc_top1
                top_result_name_top1 = args.experiment_name + "/checkpoint_" + checkpoint_id + "_" + str(i)

                # === Top K ===
            _, _, eval_acc_topk = evaluate_top_k(tested_model, eval_loader, device)

            if eval_acc_topk > top_result_topk:
                top_result_topk = eval_acc_topk
                top_result_name_topk = args.experiment_name + "/checkpoint_" + checkpoint_id + "_" + str(i)

            logger(
                    f"checkpoint_{checkpoint_id}_{i}  ->  "
                    f"Top 1: {eval_acc_top1:<8} | "
                    f"Top 5: {eval_acc_topk:<8}"
                )

    path_to_load = "out-checkpoints/" + top_result_name_top1 + '.pth'
    if os.path.exists(path_to_load):
        model_top=torch.load(path_to_load, weights_only=False)

        logger('\n=== Parameter statistics ===')
        calc_total_params(model_top)

        model_top.to(device)
        model_top.eval()

        logger('\n=== Number of early exits ===')

        if train_loader:
            train_exited, train_total, train_ratio = compute_early_exit_stats(model_top, train_loader, device)
            logger(f"[Train] {train_exited}/{train_total} | {train_ratio:.2%}")
            logger()
        if val_loader:
            val_exited, val_total, val_ratio = compute_early_exit_stats(model_top, val_loader, device)
            logger(f"[Val]   {val_exited}/{val_total} | {val_ratio:.2%}")
            logger()

        if eval_loader:
            test_exited, test_total, test_ratio = compute_early_exit_stats(model_top, eval_loader, device)
            logger(f"[Test]  {test_exited}/{test_total} | {test_ratio:.2%}")
            logger()

    logger("\nThe top result was recorded at " + str(
            top_result_top1) + " testing accuracy. The best checkpoint is " + top_result_name_top1 + ".")


Testing checkpointed models starting...

checkpoint_t_0  ->  Top 1: 0.3663522012578616 | Top 5: 0.7688679245283019
checkpoint_t_1  ->  Top 1: 0.9858490566037735 | Top 5: 0.9968553459119497
checkpoint_t_2  ->  Top 1: 0.9937106918238994 | Top 5: 0.9968553459119497
checkpoint_t_3  ->  Top 1: 0.9937106918238994 | Top 5: 0.9984276729559748
checkpoint_t_4  ->  Top 1: 0.9968553459119497 | Top 5: 1.0     
checkpoint_t_5  ->  Top 1: 0.9952830188679245 | Top 5: 0.9984276729559748
checkpoint_t_6  ->  Top 1: 0.9968553459119497 | Top 5: 1.0     
checkpoint_t_7  ->  Top 1: 0.9968553459119497 | Top 5: 1.0     
checkpoint_t_8  ->  Top 1: 0.9968553459119497 | Top 5: 1.0     
checkpoint_t_9  ->  Top 1: 0.9984276729559748 | Top 5: 1.0     
checkpoint_t_10  ->  Top 1: 0.9968553459119497 | Top 5: 1.0     

=== Parameter statistics ===
Total params: 4,120,045
Trainable params: 4,120,045
Frozen params: 0

=== Number of early exits ===
ENCODER
Exit in 1 stream 72
Exit in 2 stream 0
Exit in 3 stream 0
Full de

In [ ]:
if args.plot_stats:
    fig, ax = plt.subplots()
    ax.plot(range(1, len(losses) + 1), losses, c="#D64436", label="Training loss")
    ax.plot(range(1, len(train_accs) + 1), train_accs, c="#00B09B", label="Training accuracy")

    if val_loader:
        ax.plot(range(1, len(val_accs) + 1), val_accs, c="#E0A938", label="Validation accuracy")

    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    ax.set(xlabel="Epoch", ylabel="Accuracy / Loss", title="")
    plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.05), ncol=4, fancybox=True, shadow=True,
                   fontsize="xx-small")
    ax.grid()

    fig.savefig("out-img/" + args.experiment_name + "_loss.png")

In [ ]:
if args.plot_lr:
    fig1, ax1 = plt.subplots()
    ax1.plot(range(1, len(lr_progress) + 1), lr_progress, label="LR")
    ax1.set(xlabel="Epoch", ylabel="LR", title="")
    ax1.grid()

    fig1.savefig("out-img/" + args.experiment_name + "_lr.png")

    # PLOT 2: Training time
if args.record_training_time:
    fig1, ax2 = plt.subplots()
    ax2.plot(range(1, len(avg_train_time_sec_list) + 1), avg_train_time_sec_list,
                 label="AVG Training Time per sample")
    ax2.set(xlabel="Epoch", ylabel="Second", title="")
    ax2.grid()

    fig1.savefig("out-img/" + args.experiment_name + "_tt.png")
logger("Experiment parameters:")
logger(f"\t\t\tnum_com_layers={args.num_com_layers}")
logger(f"\t\t\tnum_enc_layers={args.num_enc_layers} | pat_enc={args.pat_enc}")
logger(f"\t\t\tnum_dec_layers={args.num_dec_layers} | pat_enc={args.pat_dec}")
logger("\nAny desired statistics have been plotted.\nThe experiment is finished.")